# E-Commerce Customer Churn Analysis
This notebook demonstrates end-to-end data loading, cleaning, exploratory data analysis, feature engineering, and model preparation for customer churn prediction.


In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import plotly.express as px
from src.data_cleaning import load_data, clean_data, summarize_data
from src.feature_engineering import engineer_features

pd.set_option('display.max_columns', None)

raw_df = load_data('../dataset/customer_churn.csv')
print('Shape:', raw_df.shape)
print('Data Types:')
print(raw_df.dtypes)
print('\nMissing values:')
print(raw_df.isnull().sum())
print('\nDuplicate rows:', raw_df.duplicated().sum())
raw_df.head()


In [ ]:
clean_df = clean_data(raw_df)
print('Cleaned Shape:', clean_df.shape)
print('Duplicates after cleaning:', clean_df.duplicated().sum())
summary = summarize_data(clean_df)
summary.head(12)


In [ ]:
clean_df['complaintflag'] = clean_df['complaint'].astype(str).str.lower().isin(['yes', 'y', '1', 'true']).astype(int)
clean_df['order_frequency'] = clean_df['ordercount'] / clean_df['tenure'].replace(0, 1)
clean_df['customervaluescore'] = clean_df['ordercount'] * clean_df['cashbackamount']
clean_df['loyaltycategory'] = clean_df['tenure'].apply(lambda x: 'New' if x <= 12 else ('Regular' if x <= 24 else 'Loyal'))
median_distance = clean_df['warehousetohome'].median()
clean_df['highdistancecustomer'] = (clean_df['warehousetohome'] > median_distance).astype(int)
clean_df['recentcustomeractivity'] = clean_df['daysincelastorder'].apply(
    lambda x: 'Very Active' if x <= 7 else ('Active' if x <= 30 else ('At Risk' if x <= 60 else 'Dormant'))
)
clean_df['engagementscore'] = clean_df['hourspendonapp'] * clean_df['ordercount']
clean_df[['order_frequency', 'customervaluescore', 'loyaltycategory', 'highdistancecustomer', 'recentcustomeractivity', 'engagementscore']].head()


In [ ]:
plt.figure(figsize=(8, 5))
sns.countplot(data=clean_df, x='churn', palette=['#2ecc71', '#e74c3c'])
plt.title('Churn Distribution')
plt.xlabel('Churn')
plt.ylabel('Number of Customers')
plt.xticks([0, 1], ['Stay', 'Churn'])
plt.show()

fig = px.histogram(clean_df, x='tenure', color='churn', barmode='group', title='Tenure vs Churn')
fig.show()

fig = px.box(clean_df, x='churn', y='ordercount', color='churn', title='Order Count vs Churn')
fig.show()

fig = px.box(clean_df, x='churn', y='satisfactionscore', color='churn', title='Satisfaction Score vs Churn')
fig.show()

fig = px.box(clean_df, x='churn', y='cashbackamount', color='churn', title='Cashback Amount vs Churn')
fig.show()


In [ ]:
plt.figure(figsize=(8, 5))
sns.countplot(data=clean_df, x='complaintflag', palette=['#3498db', '#e74c3c'])
plt.title('Complaint Analysis')
plt.xlabel('Complaint Flag')
plt.ylabel('Count')
plt.xticks([0, 1], ['No Complaint', 'Complaint'])
plt.show()

plt.figure(figsize=(12, 10))
sns.heatmap(clean_df.corr(), annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Correlation Heatmap')
plt.show()

fig = px.bar(clean_df.groupby('preferredpaymentmode')['churn'].mean().reset_index(),
             x='preferredpaymentmode', y='churn', title='Payment Mode Analysis', labels={'churn': 'Churn Rate'})
fig.show()

fig = px.bar(clean_df.groupby('preferredlogindevice')['churn'].mean().reset_index(),
             x='preferredlogindevice', y='churn', title='Device Analysis', labels={'churn': 'Churn Rate'})
fig.show()

fig = px.bar(clean_df.groupby('preferedordercat')['churn'].mean().reset_index(),
             x='preferedordercat', y='churn', title='Customer Category Analysis', labels={'churn': 'Churn Rate'})
fig.show()


## Insights

- Customers with low satisfaction and high days since last order have higher churn rates.
- Complaint customers are significantly more likely to churn.
- Payment mode and preferred device impact customer retention.
- Loyalty category and engagement score are strong features for model building.
